# CGM Preprocessing

* this notebook produces the corresponding preprocessing for CGM data from CGMacros.

In [23]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('..').resolve()

DATA_ROOT = PROJECT_ROOT
print(f"DATA_ROOT: {DATA_ROOT}")

DATA_ROOT: /Users/ucabtt4/Projects/Dissertation/dissertation


In [24]:
import os
import json
import pickle

import numpy as np
import pandas as pd

In [25]:
# -- Pipeline configuration --
DATA_ROOT = '..'
NATIVE_STEP_MIN   = 5     # Dexcom's true native sampling interval
WINDOW_MINUTES    = 120
WINDOW_STEPS      = WINDOW_MINUTES // NATIVE_STEP_MIN
HORIZON_MINUTES   = [15, 30, 60, 90, 120]
HORIZON_STEPS     = [h // NATIVE_STEP_MIN for h in HORIZON_MINUTES]  # = [3, 6, 12, 18, 24]
STEP_STEPS        = 1     # slide by 1 native point (= 5 minutes) between consecutive samples
MAX_GAP_STEPS      = 3     # interpolate at most 3 consecutive missing native points (~15 min)

EXCLUDE_PIDS = {'CGMacros-007', 'CGMacros-018', 'CGMacros-026'}

OUT_DIR = os.path.join('..','preprocessed_data','processed_5min')
os.makedirs(OUT_DIR, exist_ok=True)

In [26]:
def load_full_grid(pid):
    """Load a participant's Dexcom glucose data onto a complete 1-minute grid while preserving internal missing values."""
    df = pd.read_csv(os.path.join(DATA_ROOT, 'CGMacros', pid, f'{pid}.csv'))
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df = df.set_index('Timestamp').sort_index()
    gl = df['Dexcom GL']
    valid = gl.dropna()
    full_idx = pd.date_range(valid.index.min(), valid.index.max(), freq='1min')
    return gl.reindex(full_idx)

def find_breakpoints(gl_1min):
    """Identify slope-change positions in the 1-minute glucose series using non-zero second differences."""
    d2 = gl_1min.diff().diff()
    return d2.abs() > 1e-6

def find_real_readings(gl_1min):
    """Identify genuine Dexcom reading positions by shifting detected slope-change positions back by one minute."""
    return find_breakpoints(gl_1min).shift(-1, fill_value=False)

def dominant_phase(gl_1min, min_real=10):
    """Determine the participant's dominant 5-minute sampling phase from the detected genuine Dexcom readings."""
    is_real = find_real_readings(gl_1min)
    if is_real.sum() < min_real:
        return None, 0.0
    counts = (gl_1min.index[is_real].minute % 5).value_counts(normalize=True)
    return int(counts.index[0]), float(counts.iloc[0])

In [27]:
# extract the 5-minute series from 1-minute series
def extract_native_5min(gl_1min, phase):
    native_index = gl_1min.index[gl_1min.index.minute % 5 == phase]
    native_index = pd.date_range(native_index.min(), native_index.max(),
                                  freq=f'{NATIVE_STEP_MIN}min')
    return gl_1min.reindex(native_index)

# Demonstrate on CGMacros-001
pid_demo = 'CGMacros-001'
gl1_demo = load_full_grid(pid_demo)
phase_demo, purity_demo = dominant_phase(gl1_demo)
native_demo = extract_native_5min(gl1_demo, phase_demo)

print(f"{pid_demo}: phase={phase_demo}  purity={purity_demo:.1%}")
print(f"1-minute grid: {len(gl1_demo)} rows, {gl1_demo.notna().sum()} non-null")
print(f"native 5-minute grid: {len(native_demo)} rows, {native_demo.notna().sum()} non-null "
      f"({native_demo.isna().sum()} genuine gaps)")
native_demo.head(10)

CGMacros-001: phase=4  purity=100.0%
1-minute grid: 14275 rows, 14265 non-null
native 5-minute grid: 2855 rows, 2853 non-null (2 genuine gaps)


2020-05-01 11:39:00    141.0
2020-05-01 11:44:00    159.0
2020-05-01 11:49:00    173.0
2020-05-01 11:54:00    176.0
2020-05-01 11:59:00    165.0
2020-05-01 12:04:00    142.0
2020-05-01 12:09:00    112.0
2020-05-01 12:14:00     87.0
2020-05-01 12:19:00     86.0
2020-05-01 12:24:00     91.0
Freq: 5min, Name: Dexcom GL, dtype: float64

In [28]:
# Gap-limited Linear Interpolation on Native Grid
def interpolate_native(native_series, max_gap_steps=MAX_GAP_STEPS):
    return native_series.interpolate(method='linear', limit=max_gap_steps,
                                      limit_direction='forward')

# Demonstrate again on CGMacros-001
native_demo_interp = interpolate_native(native_demo)
before = native_demo.isna().sum()
after = native_demo_interp.isna().sum()
print(f"{pid_demo}: {before} missing native points before gap-limited interpolation, "
      f"{after} after (gap limit = {MAX_GAP_STEPS} native steps = "
      f"{MAX_GAP_STEPS * NATIVE_STEP_MIN} minutes)")

CGMacros-001: 2 missing native points before gap-limited interpolation, 0 after (gap limit = 3 native steps = 15 minutes)


In [29]:
# create samples based on 5-minute
def create_samples(native_series, df_raw, pid, participant_dir):
    samples = []
    max_horizon_steps = max(HORIZON_STEPS)
    n = len(native_series)

    for i in range(0, n - WINDOW_STEPS - max_horizon_steps, STEP_STEPS):
        cgm_window = native_series.iloc[i: i + WINDOW_STEPS].values
        if np.isnan(cgm_window).any():
            continue

        t_end = native_series.index[i + WINDOW_STEPS - 1]

        targets = {}
        valid = True
        for h_min, h_steps in zip(HORIZON_MINUTES, HORIZON_STEPS):
            target_idx = i + WINDOW_STEPS - 1 + h_steps
            if target_idx >= n:
                valid = False
                break
            target_val = native_series.iloc[target_idx]
            if np.isnan(target_val):
                valid = False
                break
            targets[h_min] = float(target_val)
        if not valid:
            continue

        # image_path, time_since_meal = get_meal_info(df_raw, t_end, participant_dir)

        samples.append({
            'participant_id':  pid,
            'timestamp':       t_end,
            'cgm_sequence':    cgm_window.astype(np.float32),
            # 'image_path':      image_path,
            # 'has_image':       image_path is not None,
            # 'time_since_meal': time_since_meal,
            **{f'target_{h}': targets[h] for h in HORIZON_MINUTES},
        })
    return samples

In [30]:
# create all the samples except 007, 018, 026
cgm_root = os.path.join(DATA_ROOT, 'CGMacros')
all_pids = sorted(d for d in os.listdir(cgm_root) if d.startswith('CGMacros-'))
kept_pids = [p for p in all_pids if p not in EXCLUDE_PIDS]

print(f"Total participants: {len(all_pids)}  Excluded: {sorted(EXCLUDE_PIDS)}  "
      f"Kept: {len(kept_pids)}")

all_samples = []
phase_report = []

for pid in kept_pids:
    pdir = os.path.join(cgm_root, pid)
    gl1 = load_full_grid(pid)
    phase, purity = dominant_phase(gl1)
    phase_report.append({'participant_id': pid, 'phase': phase, 'purity': purity})
    if phase is None or purity < 0.95:
        print(f"  !! {pid}: phase detection failed or unstable (purity={purity:.2f}) -- skipping")
        continue

    native = extract_native_5min(gl1, phase)
    native = interpolate_native(native)

    df_raw = pd.read_csv(os.path.join(pdir, f'{pid}.csv'))
    df_raw['Timestamp'] = pd.to_datetime(df_raw['Timestamp'])
    df_raw = df_raw.set_index('Timestamp').sort_index()

    all_samples.extend(create_samples(native, df_raw, pid, pdir))

phase_report = pd.DataFrame(phase_report)
print(f"\nAll {len(kept_pids)} kept participants: phase purity range "
      f"[{phase_report['purity'].min():.3f}, {phase_report['purity'].max():.3f}]")
print(f"Total samples: {len(all_samples):,}")
# n_img = sum(s['has_image'] for s in all_samples)
# print(f"Samples with image: {n_img:,} ({100*n_img/len(all_samples):.1f}%)")

Total participants: 45  Excluded: ['CGMacros-007', 'CGMacros-018', 'CGMacros-026']  Kept: 42

All 42 kept participants: phase purity range [1.000, 1.000]
Total samples: 117,113


In [31]:
# participant-level split to train, valid, test groups (30 / 6 / 6) -- (5: 1: 1)
bio = pd.read_csv(os.path.join(DATA_ROOT, 'CGMacros','bio.csv'))
bio['participant_id'] = bio['subject'].apply(lambda s: f'CGMacros-{s:03d}')

def glycemic_group(a1c):
    if a1c < 5.7:
        return 'Healthy'
    elif a1c < 6.5:
        return 'Prediabetes'
    return 'Type2Diabetes'

bio['group'] = bio['A1c PDL (Lab)'].apply(glycemic_group)
bio = bio[bio['participant_id'].isin(kept_pids)].set_index('participant_id')

print(bio['group'].value_counts())
assert set(bio['group'].value_counts()) == {14}, "expected exactly 14 per group after exclusions"

# Within each group: sort by the participant's own mean CGM value (from the samples just built),
# then take 2 for test, 2 for val, 10 for train -- spread across the sorted range rather than
# grabbing adjacent extremes.
pid_mean_gl = {}
for s in all_samples:
    pid_mean_gl.setdefault(s['participant_id'], []).append(s['cgm_sequence'].mean())
pid_mean_gl = {p: np.mean(v) for p, v in pid_mean_gl.items()}

train_pids, val_pids, test_pids = set(), set(), set()
for grp in ['Healthy', 'Prediabetes', 'Type2Diabetes']:
    grp_pids = sorted(bio.index[bio['group'] == grp], key=lambda p: pid_mean_gl[p])
    # 14 participants, ascending by mean glucose -> take evenly spread pairs for val/test
    val_pids.update([grp_pids[4], grp_pids[9]])
    test_pids.update([grp_pids[1], grp_pids[12]])
    train_pids.update(p for p in grp_pids if p not in val_pids and p not in test_pids)

splits = {'train': [], 'val': [], 'test': []}
for s in all_samples:
    if s['participant_id'] in train_pids:
        splits['train'].append(s)
    elif s['participant_id'] in val_pids:
        splits['val'].append(s)
    elif s['participant_id'] in test_pids:
        splits['test'].append(s)

for name, pids_ in [('train', train_pids), ('val', val_pids), ('test', test_pids)]:
    grp_counts = bio.loc[list(pids_), 'group'].value_counts().to_dict()
    print(f"{name:5s}: {len(pids_)} participants {grp_counts}, {len(splits[name]):,} samples")

group
Healthy          14
Type2Diabetes    14
Prediabetes      14
Name: count, dtype: int64
train: 30 participants {'Prediabetes': 10, 'Healthy': 10, 'Type2Diabetes': 10}, 83,532 samples
val  : 6 participants {'Prediabetes': 2, 'Healthy': 2, 'Type2Diabetes': 2}, 16,739 samples
test : 6 participants {'Healthy': 2, 'Type2Diabetes': 2, 'Prediabetes': 2}, 16,842 samples


In [32]:
train_samples = splits['train']
seqs = np.concatenate([s['cgm_sequence'] for s in train_samples])
targets = np.array([[s[f'target_{h}'] for h in HORIZON_MINUTES] for s in train_samples]).ravel()
all_vals = np.concatenate([seqs, targets])

cgm_stats = {'cgm_mean': float(all_vals.mean()), 'cgm_std': float(all_vals.std())}
print(f"cgm_mean = {cgm_stats['cgm_mean']:.2f} mg/dL, cgm_std = {cgm_stats['cgm_std']:.2f} mg/dL")

cgm_mean = 140.10 mg/dL, cgm_std = 42.13 mg/dL


In [33]:
for name, split_samples in splits.items():
    with open(os.path.join(OUT_DIR, f'{name}.pkl'), 'wb') as f:
        pickle.dump(split_samples, f)

with open(os.path.join(OUT_DIR, 'cgm_stats.json'), 'w') as f:
    json.dump(cgm_stats, f, indent=2)

with open(os.path.join(OUT_DIR, 'split_participants.json'), 'w') as f:
    json.dump({'train': sorted(train_pids), 'val': sorted(val_pids),
               'test': sorted(test_pids), 'excluded': sorted(EXCLUDE_PIDS)}, f, indent=2)

print(f"Saved train/val/test .pkl + cgm_stats.json + split_participants.json -> {OUT_DIR}")

Saved train/val/test .pkl + cgm_stats.json + split_participants.json -> ../preprocessed_data/processed_5min
